# F7 — Latency, throughput and cost on a single T4

Measures what one document actually costs: single-stream **p50/p95 milliseconds**, an
amortized **batch-size sweep**, and a derived **$/1k documents**. Writes
`results/bench/<arm>.json` and `results/bench/<arm>_sweep.json` for all four GPU arms.

**Before running:**

1. Settings → Accelerator → **`GPU T4 x2`** (the code uses `cuda:0` only — SPEC §2.2).
2. Settings → Internet **on**.
3. Add Data → the **`sxl-data`** dataset (`train.jsonl`, `eval_gold.jsonl`).
4. Add-ons → Secrets → add **`HF_TOKEN`** (read scope is enough — this notebook
   publishes nothing). Needed only if the adapter repo is private; it also avoids
   anonymous download rate-limits. The two base arms run without it.
5. Set `COMMIT_SHA` and `ADAPTER_REPO` in the cells below.

**All four arms run in one session on purpose** — they must share hardware, thermal state and
driver version, or the numbers are not comparable with each other.

### Sample size: 10 documents × 3 repeats

`config.BENCH_N_DOCS` is 100; this run overrides it to **10** to fit the remaining GPU
quota. That is a deliberate, recorded trade — `n_docs` is written into every bench file, so
no reader has to guess — but it changes what two numbers mean, and F8 must say so:

- **`p95_ms` over 10 samples is the slowest of 10**, not a tail estimate. By nearest rank,
  p95 of 10 values *is* the maximum. Quote it as "slowest of 10" or not at all.
- **`p50_ms` is a 10-sample median** and will move between runs. This is exactly why
  `--repeats 3` is kept: `p50_spread_pct` across the three passes is the honest measure of
  how much to trust it. A spread above ~15% means the number is noise and the notebook says so.

The batch sweep is capped at **8**: 16 and 32 cannot fill one whole batch from 10 documents,
and `measure_sweep` skips rather than mislabels them (the sweep file's `note` records this).

**What this notebook is guarding against.** The project pitch says "45 ms on a T4". SPEC §1.1
disowns that number in advance: a 1.7B model in fp16 on a T4 runs at 40–70 tok/s single-stream,
so ~150 tokens of JSON costs **seconds**. Expect a p50 between roughly 1,500 and 4,000 ms. If a
number comes back near 45 ms, that is a bug — a missing `torch.cuda.synchronize()` — and
`sxl.gpu.bench.check_plausible` will raise rather than let it reach a results table.

At 10 × 3 this should cost well under 30 minutes of the ~30 h/week quota (SPEC §6.6).


In [ ]:
# HF_HOME must be set BEFORE anything imports huggingface_hub: the cache path is
# frozen into module constants at import time. /kaggle/working is only 20 GB and is
# the persisted notebook output; /kaggle/tmp is ~60 GB of scratch (SPEC §2.2).
import os
import time

os.environ["HF_HOME"] = "/kaggle/tmp/hf"
os.makedirs("/kaggle/tmp/hf", exist_ok=True)

# HF_TOKEN from Kaggle Secrets (same pattern as F6). Set here, before anything
# imports huggingface_hub, so every later download is authenticated.
#
# It is NOT needed to make Qwen3-1.7B download faster -- that repo is ungated and
# auth does not change throughput. It is here for two real reasons:
#   1. anonymous downloads are rate-limited, and a 429 mid-fetch stalls a session;
#   2. `lora_ft` / `lora_ft_constrained` pull the adapter from the Hub, which needs
#      a token if that repo is private.
# Read-only scope is enough: this notebook publishes nothing.
try:
    from kaggle_secrets import UserSecretsClient

    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN loaded from Kaggle Secrets")
except Exception as exc:
    # Not fatal for the two base arms; the fine-tuned arms will fail later if the
    # adapter repo is private. Say so now rather than 40 minutes in.
    print(f"no HF_TOKEN ({type(exc).__name__}) — fine for base arms, may block lora_ft")

# The actual download accelerator. `hf_transfer` is a Rust multi-threaded fetcher
# and is typically several times faster than the default on Kaggle's link. Enabled
# only if it imports: setting the flag without the package installed makes
# huggingface_hub raise on the first download, which would cost a session.
try:
    import hf_transfer  # noqa: F401

    os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
    print("hf_transfer enabled")
except ImportError:
    print("hf_transfer not installed — default downloader (add `!pip install -q hf_transfer`)")

# The 3.4 GB base model is fetched ONCE and cached in /kaggle/tmp/hf for all four
# arms in this session. Download time is a one-off, not a per-arm cost.

# F7's acceptance criteria include "total GPU time under 2.5 hours", which is only
# checkable if the session clock is recorded rather than estimated afterwards.
SESSION_STARTED = time.time()
print("session started")


In [ ]:
# Pin a commit SHA, never a branch (SPEC §2.4): a mid-session push must not change
# what a running notebook is executing.
COMMIT_SHA = "0000000000000000000000000000000000000000"  # <- paste `git rev-parse HEAD`

# Shape check, not a placeholder comparison: a find-and-replace over this cell
# would rewrite both copies of a placeholder string and silently disarm the guard.
import re

assert re.fullmatch(r"[0-9a-f]{40}", COMMIT_SHA)
assert set(COMMIT_SHA) != {"0"}

REPO = "https://github.com/RazaAli1010/schema-extract-lab"

# `schema-extract-lab`, not `sxl` — that is the DISTRIBUTION name from pyproject.toml.
!pip install -q "schema-extract-lab[gpu] @ git+{REPO}@{COMMIT_SHA}"

# Same removal F6 needs: torchao/torchvision/torchaudio drag in a torch pin that
# fights peft on the Kaggle base image.
!pip uninstall -q -y torchvision torchaudio torchao

# >>> RESTART THE SESSION after this cell, then skip it on the re-run. <<<


In [ ]:
# Version banner (SPEC §5.7). Printed into the saved output so a stale Kaggle base
# image is visible in the artifact rather than being a mystery six weeks later.
import importlib.metadata as md

import torch

for pkg in (
    "torch", "transformers", "trl", "peft", "accelerate",
    "bitsandbytes", "outlines", "schema-extract-lab",
):
    try:
        print(f"{pkg:>20} {md.version(pkg)}")
    except md.PackageNotFoundError:
        print(f"{pkg:>20} NOT INSTALLED")

name = torch.cuda.get_device_name(0)
capability = torch.cuda.get_device_capability(0)
print(f"\n{name}  capability={capability}  n_gpus={torch.cuda.device_count()}")

# Kaggle's `GPU T4 x2` and `GPU P100` are DIFFERENT HARDWARE. A P100 session would
# produce numbers that are not comparable across arms and nothing in the output
# would say so. `sxl.gpu.bench.assert_t4` repeats this check per run; failing here
# costs a second instead of an hour.
assert capability == (7, 5), f"expected a Turing T4 (7,5), got {name} {capability}"
assert "T4" in name, name

!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv


In [ ]:
# The package is a pip install here, so `config.ROOT` points into site-packages and
# every default path is wrong. Pass all of them explicitly (see config.py).
#
# The mount point is discovered rather than hard-coded: depending on how a dataset
# is attached, Kaggle mounts it at /kaggle/input/<slug>/ OR at
# /kaggle/input/datasets/<owner>/<slug>/, and guessing wrong costs a session.
import glob
import json
import math

matches = glob.glob("/kaggle/input/**/eval_gold.jsonl", recursive=True)
assert matches, "sxl-data is not attached — use Add Input in the sidebar"
assert len(matches) == 1, f"eval_gold.jsonl found in several places: {matches}"

DATA = os.path.dirname(matches[0])
GOLD = f"{DATA}/eval_gold.jsonl"
TRAIN = f"{DATA}/train.jsonl"
OUT = "/kaggle/working/bench"
os.makedirs(OUT, exist_ok=True)

# The adapter published by F6, loaded from the Hub. `lora_ft` and
# `lora_ft_constrained` are benchmarked with the SAME adapter that produced their
# accuracy numbers, or the two tables describe different models.
ADAPTER_REPO = "razaali1607/qwen3-1.7b-jobpost-lora"

print(f"gold={GOLD}\ntrain={TRAIN}\nout={OUT}\nadapter={ADAPTER_REPO}")


In [ ]:
# PROBE: 3 documents, 1 repeat, no sweep. Under a minute.
#
# Also does the one-off model download, so the elapsed times printed by the arm
# cells below are benchmark time, not fetch time.
N_DOCS = 10        # per pass, per arm
REPEATS = 3        # kept at 3: the spread across repeats is what makes a small
                   # sample honest rather than a single lucky reading
BATCH_SIZES = "1,2,4,8"  # 16 and 32 cannot fill a whole batch from 10 documents

!sxl gpu bench --arm base_fewshot --gold {GOLD} --train {TRAIN} --n-docs 3 --warmup 2 --repeats 1 --batch-sizes 1 --out-dir /kaggle/working/_probe

probe = json.load(open("/kaggle/working/_probe/base_fewshot.json"))
print(json.dumps(probe, indent=2))

# Projection for the plan below: 4 arms x N_DOCS x REPEATS, single-stream only.
# The sweep adds roughly 30-50% on the two unconstrained arms, and the constrained
# arms are slower per token than this probe (Outlines masks logits every step), so
# treat this as a FLOOR.
floor_min = probe["p50_ms"] * N_DOCS * REPEATS * 4 / 1000 / 60
print(f"\np50 = {probe['p50_ms']:.0f} ms over {probe['mean_completion_tokens']:.0f} tokens")
print(f"projected single-stream total for 4 arms: ~{floor_min:.0f} min (a FLOOR)")

# The honest-measurement guard, restated where a reader sees it.
assert probe["p50_ms"] > probe["mean_completion_tokens"] * 2, "implausibly fast — missing sync?"


In [ ]:
# The two prompted baseline arms. `base_fewshot` is the competitor that matters
# (SPEC §3.6); `base_fewshot_constrained` is the same prompt through Outlines, whose
# logit masking has real per-token overhead — measuring that overhead is the point.
#
# --warmup 5, not the default 10: at N_DOCS=10 a 10-iteration warmup would cost as
# much GPU time as the measurement itself. 5 is still enough to absorb the one-time
# CUDA context and first-sdpa costs, which is all warmup is for.
#
# One long line, not a backslash continuation: `!` cells are handed to the shell by
# IPython and line-joining behaviour there is not worth betting a session on.
!sxl gpu bench --arm base_fewshot --gold {GOLD} --train {TRAIN} --n-docs {N_DOCS} --repeats {REPEATS} --warmup 5 --batch-sizes {BATCH_SIZES} --out-dir {OUT}

!sxl gpu bench --arm base_fewshot_constrained --gold {GOLD} --train {TRAIN} --n-docs {N_DOCS} --repeats {REPEATS} --warmup 5 --batch-sizes {BATCH_SIZES} --out-dir {OUT}

print(f"elapsed so far: {(time.time() - SESSION_STARTED) / 60:.0f} min")


In [ ]:
# The fine-tuned arms, adapter loaded from the Hub. No --train flag: these prompts
# have no exemplars, and the absence of that file is what proves the branch works.
#
# Identical --n-docs / --repeats / --warmup / --batch-sizes to the cell above. That
# is not tidiness: the four arms are only comparable if they measured the same
# documents the same number of times on the same card.
#
# `mean_completion_tokens` is the number to watch here. The fine-tune should emit
# tighter JSON than the few-shot baseline (no preamble, learned key order), and if
# lora_ft is faster, THAT is why — not the adapter being intrinsically quicker. An
# unmerged LoRA adds two small matmuls per target module per token; it is slower per
# token, not faster (F7 §Implementation notes). F8 must attribute this correctly.
!sxl gpu bench --arm lora_ft --adapter {ADAPTER_REPO} --gold {GOLD} --n-docs {N_DOCS} --repeats {REPEATS} --warmup 5 --batch-sizes {BATCH_SIZES} --out-dir {OUT}

!sxl gpu bench --arm lora_ft_constrained --adapter {ADAPTER_REPO} --gold {GOLD} --n-docs {N_DOCS} --repeats {REPEATS} --warmup 5 --batch-sizes {BATCH_SIZES} --out-dir {OUT}

print(f"elapsed so far: {(time.time() - SESSION_STARTED) / 60:.0f} min")


In [ ]:
# Acceptance criteria, asserted in the artifact itself rather than eyeballed.
ARMS = ("base_fewshot", "base_fewshot_constrained", "lora_ft", "lora_ft_constrained")

print(f"{'arm':<26} {'p50 ms':>9} {'p95 ms':>9} {'tok':>7} {'spread':>7} "
      f"{'docs/s':>8} {'$/1k':>9}")
for arm in ARMS:
    d = json.load(open(f"{OUT}/{arm}.json"))

    # Hard contract — these are what F8's table depends on, so they stay assertions.
    assert "T4" in d["gpu_name"], (arm, d["gpu_name"])
    assert d["dtype"] == "float16", (arm, d["dtype"])
    assert d["measurement"] == "local_gpu", arm
    # The plausibility guard, re-checked on the file that will actually be committed.
    assert d["p50_ms"] > d["mean_completion_tokens"] * 2, ("implausible", arm, d["p50_ms"])
    # No accuracy metric may appear in results/bench/ — F4 owns those (F7 §Out of scope).
    assert not {"macro_f1", "schema_valid_rate"} & set(d), arm

    print(f"{d['arm']:<26} {d['p50_ms']:>9.1f} {d['p95_ms']:>9.1f} "
          f"{d['mean_completion_tokens']:>7.1f} {d['p50_spread_pct']:>6.1f}% "
          f"{d['throughput_docs_per_s']:>8.3f} {d['cost_per_1k_docs_usd']:>9.5f}")

    # Kaggle GPUs are shared and thermally variable. A wide spread is reportable,
    # not fatal — but quoting the best of three would be dishonest, so F8 gets told.
    if d["p50_spread_pct"] > 15.0:
        print(f"    NOTE: p50 varied {d['p50_spread_pct']:.1f}% across repeats — F8 must say so")
    if d["index_build_s"]:
        print(f"    outlines index build: {d['index_build_s']:.1f}s (one-time, excluded above)")

# The amortized numbers live in a SEPARATE table on purpose. Batching does not make
# one document faster; it makes the GPU busier. Conflating the two is the failure
# F7 exists to prevent (SPEC §1.1).
print(f"\n{'arm':<26} {'best bs':>8} {'docs/s':>8} {'ms/doc':>9} {'$/1k':>9} {'VRAM GB':>8}")
for arm in ARMS:
    s = json.load(open(f"{OUT}/{arm}_sweep.json"))
    peak = max(e["peak_vram_gb"] for e in s["sweep"])
    print(f"{s['arm']:<26} {s['best_batch_size']:>8} {s['best_throughput_docs_per_s']:>8.3f} "
          f"{s['best_amortized_ms_per_doc']:>9.1f} {s['best_cost_per_1k_docs_usd']:>9.5f} "
          f"{peak:>8.2f}")
    oomed = [e["batch_size"] for e in s["sweep"] if e["oom"]]
    if oomed:
        print(f"    OOM at batch {oomed} — recorded, not hidden")
    if s["note"]:
        print(f"    {s['note']}")


In [ ]:
# Copy to the persisted notebook output and record the GPU time this feature cost.
!mkdir -p /kaggle/working/bench_out
!cp {OUT}/*.json /kaggle/working/bench_out/
!ls -la /kaggle/working/bench_out/

minutes = (time.time() - SESSION_STARTED) / 60
print(f"\ntotal session time: {minutes:.0f} min ({minutes / 60:.2f} h)")
print("F7 budgets 2.5 h. Record this number in the run log — it is an acceptance criterion.")
if minutes > 150:
    print("OVER BUDGET — report the real figure rather than the budgeted one (SPEC §1.1).")


## Back on the laptop

Download from this notebook's output:

- `bench_out/*.json` → `results/bench/`

Then, on the laptop (network only, no GPU — it measures the hosted teacher over HTTP):

```bash
sxl bench teacher --n 30    # this run used 30; config.BENCH_TEACHER_N defaults to 50
pytest -q
git add results/bench && git commit -m "F7: measured latency/throughput/cost on a T4"
```

**Read the numbers before quoting them.**

- **Sample sizes differ between the GPU arms and the teacher, and both are small.** The GPU
  arms are 10 documents × 3 repeats; the teacher is **30 calls × 1 pass**. Every file records
  its own `n_docs`, so F8 must not present them as equally well-sampled. The teacher has no
  repeats at all, so its `p50_spread_pct` is `0.0` because none was measured — *not* because
  it was stable.
- **`p95_ms` is the maximum at these sample sizes.** By nearest rank, p95 of 10 values is the
  10th. On `base_fewshot` that lands on a 35.8 s outlier against an 8.9 s median — real, but
  "slowest of 10", not a tail estimate.
- **Single-stream p50 and amortized-at-best-batch are different quantities** and live in
  different files. `p50_ms` is what one user waits for; `best_amortized_ms_per_doc` is what a
  batch pipeline pays per document. F8 must label them separately — never merge the columns.
- **`best_batch_size` is the largest that did not OOM, which is not always the fastest.**
  On `base_fewshot` throughput *peaks at batch 4* and falls at 8 (padding waste and straggler
  dominance: the whole batch decodes until its longest sequence finishes). F8 should read the
  full `sweep` array rather than trusting `best_*` as a maximum.
- **The sweep stops at batch 8** because 10 documents cannot fill a wider batch. That is a
  limit of this run, not of the hardware — do not report it as "throughput saturates at 8".
- **`git_sha` is `"unknown"` in the GPU files.** The package is a pip install on Kaggle with
  no `.git`, which `config.git_sha()` handles by design. Provenance for those files is the
  pinned `COMMIT_SHA` in cell 2 — record it in the run log, because the artifact cannot.
- **`$/1k docs` rests on an assumption**, `T4_HOURLY_USD = 0.35` (~GCP on-demand). It is
  emitted into every file as `gpu_hourly_usd` so a reader on AWS `g4dn.xlarge` (~$0.53/h)
  rescales in one multiplication. Do not present it as measured.
- **`teacher.json` is not comparable with the GPU arms.** It carries
  `"measurement": "api_wall_clock"` — a network round-trip against a local `generate()` — and
  is priced at standard, not Batch, rates because a latency-sensitive deployment cannot wait
  24 hours.
- **vLLM would improve these throughput numbers substantially** on an A10/L4/A100 and is the
  correct production serving path. It is banned here because Turing (sm_75) support is
  degrading (SPEC §5.3). F8 says so plainly rather than presenting `transformers.generate()`
  as the ceiling.

**On the fine-tune being faster:** it is, and `mean_completion_tokens` says why — 105 tokens
against the baseline's 221. That is the fine-tune emitting tighter JSON, **not** the adapter
being intrinsically quicker; an unmerged LoRA adds compute per token. F8 must attribute it to
output length (SPEC §1.1).

**Now stop the session** — Kaggle bills the quota until you do.
